In [ ]:
import numpy as np
import pandas as pd
import xsim
import xtools as xt
import matplotlib.pyplot as plt
from pyqubo import Array, Constraint
from openjij import SQASampler, SASampler
from typing import Callable, Dict, List

from atm.flight.generator import ScenarioGenerator, ScenarioIterator
from atm.optimizer.objective import get_obj_single_runway, calc_assign_time
from atm.separation import recat, faa

In [ ]:
cf = xt.Config(dict(
    num=10,
    scenario=dict(
        interval=60,
        window=200,
        mode="mix",
        standard="recat",
        seed=0,
    ),
    penalty_coef=10,
))
sep = recat.TBS if cf.scenario.standard == "recat" else faa.TBS

In [ ]:
class Problem:
    
    def __init__(
            self,
            config,
            seed=None,
    ):
        if seed is not None:
            if hasattr(config.scenario, "seed"):
                config.scenario.seed = seed
            else:
                config.scenario.set("seed", seed)
        self.cf = cf = config
        self.gen = ScenarioGenerator(cf.scenario)
        self.sep = recat.TBS if cf.scenario.standard == "recat" else faa.TBS
        self._vols = self.gen(cf.num)
        self._obj = get_obj_single_runway(
            self._vols,
            sep,
            penalty_coef=cf.penalty_coef
        )
    
        self.num_vol = len(self._vols)

    def evaluate(self, x):
        return self._obj(x)
    
    @property
    def vols(self):
        return self._vols
    
    def __iter__(self):
        return ScenarioIterator(list(self._vols))

In [ ]:
prob = Problem(cf)

In [ ]:
prob.evaluate(np.arange(cf.num))

In [ ]:
cf = xt.Config(dict(
    num=4,
    scenario=dict(
        interval=30,
        window=200,
        mode="mix",
        standard="recat",
        seed=1,
    ),
    penalty_coef=10,
))
T = cf.num * 5
def efunc(e):
    return np.exp(-1. * e / T)

# initialize
prob = Problem(cf)
xc = np.arange(prob.num_vol)
ec = prob.evaluate(xc)
xb = xc
eb = ec

log = xsim.Logger()
log.store(step=0, xc=xc, ec=ec, xb=xb, eb=eb).flush()

for step in range(1, 51):
    # sample
    idx1, idx2 = np.random.choice(np.arange(prob.num_vol), 2, replace=False)
    xp = xc.copy()
    xp[idx1] = xc[idx2]
    xp[idx2] = xc[idx1]
    ep = prob.evaluate(xp)
    
    # calculate accept probability
    r = np.min([1., efunc(ep) / efunc(ec)])
    if r > np.random.rand():
        xc = xp
        ec = ep
    
    if eb >= ec:
        eb = ec
        xb = xc
    
    log.store(step=step, xc=xc, ec=ec, xb=xb, eb=eb).flush()

plt.plot(log.buffer()["step"], log.buffer()["eb"])
log.buffer()["xb"]

In [ ]:
cf = xt.Config(dict(
    num=4,
    scenario=dict(
        interval=30,
        window=200,
        mode="mix",
        standard="recat",
        seed=1,
    ),
    penalty_coef=10,
))
T = cf.num * 5
S = sep
def efunc(e):
    return np.exp(-1. * e / T)

# initialize
prob = Problem(cf)
vols = prob.vols
xc = np.arange(prob.num_vol)
ec = prob.evaluate(xc)
xb = xc
eb = ec

log = xsim.Logger()
log.store(step=0, xc=xc, ec=ec, xb=xb, eb=eb).flush()

In [ ]:
cf = xt.Config(dict(
    num=4,
    scenario=dict(
        interval=30,
        window=200,
        mode="mix",
        standard="recat",
        seed=1,
    ),
    penalty_coef=10,
))
T = cf.num * 5
S = sep
def efunc(e):
    return np.exp(-1. * e / T)

# initialize
prob = Problem(cf)
vols = prob.vols
xc = np.arange(prob.num_vol)
ec = prob.evaluate(xc)
xb = xc
eb = ec
sampler = SQASampler()

log = xsim.Logger()
log.store(step=0, xc=xc, ec=ec, xb=xb, eb=eb).flush()

print(ec)

# sample
for step in range(1, 11):
    indices_of_pairs = xc.reshape((-1, 2))
    vs = [vols[i] for i in xc]
    ks = [
        [vs[i], vs[j]]
        for i, j in indices_of_pairs
    ]
    Xs = Array.create("x", shape=len(ks), vartype="BINARY")
    lXs = list(Xs)
    cost_in = Constraint(sum([
        S(vol1, vol2) * (1 - X) + S(vol2, vol1) * X
        for(vol1, vol2), X in zip(ks, Xs)
    ]), "inner")
    cost_ex = Constraint(sum([
        (S(j2, k2) - S(j2, k1) * Xk + (S(j1, k1) - S(j2, k1)) * Xj + (S(j1, k2) - S(j2, k1)) * Xj * Xk)
        for (j1, j2), (k1, k2), Xj, Xk in zip(ks[:-1], ks[1:], lXs[:-1], lXs[1:])
    ]), "external")
    cost = cost_in + cost_ex
    model = cost.compile()
    qubo, bias = model.to_qubo()
    break
    first = sampler.sample_qubo(qubo).first
    labels = [f"x[{i}]" for i in range(len(Xs))]
    sampled_items = np.squeeze([
        paired if first.sample[lb] == 0 else paired[::-1]
        for paired, lb in zip(indices_of_pairs, labels)
    ]).flatten()
    
    eap = first.energy + bias
    eab = bias
    eap, eab
    
    xp = sampled_items
    ep = prob.evaluate(xp)
    print(xp, ep, ec)
        
    # calculate accept probability and update
    r = np.min([1., efunc(ep) * efunc(eab) / efunc(ec) / efunc(eap)])
    if r > np.random.rand():
        xc = xp
        ec = ep
    
    if eb >= ec:
        eb = ec
        xb = xc
        
    log.store(step=step, xc=xc, ec=ec, xb=xb, eb=eb).flush()

print(cost)
qubo, bias
# plt.plot(log.buffer()["step"], log.buffer()["eb"])

In [ ]:
log.buffer()["xc"]

In [ ]:
Q = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        key = (f"x[{i}]", f"x[{j}]")
        val = qubo.get(key, 0)
        Q[i, j] = val + bias
Q
# plt.imshow(Q)
# plt.colorbar()

In [ ]:
for v1 in vols:
    for v2 in vols:
        if v1 == v2:
            continue
        print(sep(v1, v2))